# 11.1 - Why RAG Exists

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

LLMs are powerful but fundamentally limited: they cannot reliably know private, current, or source-specific facts, they hallucinate plausible-looking answers, and they have a hard knowledge cutoff. Retrieval-Augmented Generation (RAG) solves this by retrieving relevant evidence before generating an answer.

## 2. Why Does This Matter?

Every enterprise LLM application (support, legal research, medical Q&A, internal knowledge bases) relies on grounding answers in real documents. Without RAG you ship answers that may be wrong, stale, or about data the model never saw.

## 3. Prerequisites

Phase 09 (GenAI), Phase 10 (LLMs).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- List the three core limitations of LLMs that RAG addresses
- Compare RAG with fine-tuning and long-context prompting
- Decide when RAG is the right architecture
- Build a tiny mock retrieve+ground demo that calls llm()

## 5. Mental Model

An LLM is a brilliant consultant who read everything up to a certain date but has no internet access and sometimes makes things up. RAG gives that consultant a library card and a search engine.

```text
Without RAG:  User Question -> LLM -> Answer (may be wrong)
With RAG:     User Question -> Retriever -> Relevant Docs -> LLM + Context -> Answer (grounded)
```


## 6. The Three Limits of an LLM
1. **Knowledge cutoff** - the model stops learning after training.
2. **Hallucination** - it invents plausible-sounding facts.
3. **Private data** - it never saw your internal documents.

RAG is the 'library card + search engine' that lets the LLM look things up at answer time instead of guessing from memory.

In [1]:
# The three problems in one table.
problems = [
    ("Knowledge cutoff", "Model only knows data up to its training date",
     "Retrieve fresh/current docs at answer time"),
    ("Hallucination", "Model confidently invents facts",
     "Ground the answer in retrieved evidence + citations"),
    ("Private / proprietary data", "The model never saw internal documents",
     "Index your own docs and retrieve before generating"),
]
for name, desc, fix in problems:
    print(f"{name:20s} | {desc:45s} | {fix}")


Knowledge cutoff     | Model only knows data up to its training date | Retrieve fresh/current docs at answer time
Hallucination        | Model confidently invents facts               | Ground the answer in retrieved evidence + citations
Private / proprietary data | The model never saw internal documents        | Index your own docs and retrieve before generating


## 7. RAG vs Fine-Tuning vs Long-Context
These are the three ways to 'give the model knowledge'. They are not interchangeable.

In [2]:
# Decision table: when to use which approach.
rows = [
    ("RAG", "Knowledge changes frequently", "Source attribution required", "Docs are private"),
    ("Fine-Tuning", "Change style/format/behaviour", "Model must LEARN a skill", "Cannot expose docs at inference"),
    ("Long-Context", "Corpus fits in the window", "Queries are simple/infrequent", "Cost/latency acceptable"),
]
print(f"{'Approach':14s} | {'Use when':35s} | {'Use when 2':35s} | {'Use when 3':25s}")
print("-" * 115)
for row in rows:
    print(f"{row[0]:14s} | {row[1]:35s} | {row[2]:35s} | {row[3]:25s}")


Approach       | Use when                            | Use when 2                          | Use when 3               
-------------------------------------------------------------------------------------------------------------------
RAG            | Knowledge changes frequently        | Source attribution required         | Docs are private         
Fine-Tuning    | Change style/format/behaviour       | Model must LEARN a skill            | Cannot expose docs at inference
Long-Context   | Corpus fits in the window           | Queries are simple/infrequent       | Cost/latency acceptable  


## 8. Decision Guidance
Rule of thumb: **use RAG** when you must cite sources, the data is private, or it changes often. **Use fine-tuning** when you need to change *behaviour/style*, not facts. **Use long-context** when the corpus is small and queries are rare.

In [3]:
def choose_approach(needs_citations, data_private_or_fresh, corpus_small, style_change):
    if needs_citations or data_private_or_fresh:
        return "RAG"
    if style_change:
        return "Fine-Tuning"
    if corpus_small:
        return "Long-Context"
    return "RAG (safest default, supports citations)"

for q in [
    ("legal brief with citations", True, True, False, False),
    ("make my assistant sound more formal", False, False, False, True),
    ("10-page handbook, simple lookup", False, False, True, False),
]:
    name, need_cit, private, small, style = q
    print(f"{name:40s} -> {choose_approach(need_cit, private, small, style)}")


legal brief with citations               -> RAG
make my assistant sound more formal      -> Fine-Tuning
10-page handbook, simple lookup          -> Long-Context


## 9. A Tiny Mock Retrieve+Ground Demo
With no API key, `llm()` returns a deterministic mock. We wire the pieces: retrieve the source doc for a query, build a grounded prompt, and let the LLM (or mock) answer.

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
import operator

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [5]:
# A pretend knowledge base (like a private corporate doc).
KNOWLEDGE = {
    "refund": "Refunds are processed within 5-7 business days after the item is received.",
    "shipping": "Standard shipping takes 5-7 business days; express takes 2-3.",
    "replacement": "Defective items are replaced free of charge within the first 30 days.",
}


def retrieve(query, k=3):
    q = query.lower()
    hits = [(score_basic(q, key), key) for key in KNOWLEDGE]
    hits.sort(reverse=True)
    return [key for _, key in hits[:k]]


def score_basic(q, key):
    return len(set(q.split()) & set(key.split()))


def answer(query):
    keys = retrieve(query)
    context = "\n".join(f"[{i+1}] {KNOWLEDGE[k]}" for i, k in enumerate(keys))
    grounded = f"Answer using ONLY the context. Cite [i].\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    return keys, grounded


query = "How fast will I get my refund?"
keys, prompt = answer(query)
print("Retrieved doc keys:", keys)
print("\n--- Grounded prompt sent to the LLM ---")
print(prompt)


Retrieved doc keys: ['shipping', 'replacement', 'refund']

--- Grounded prompt sent to the LLM ---
Answer using ONLY the context. Cite [i].
Context:
[1] Standard shipping takes 5-7 business days; express takes 2-3.
[2] Defective items are replaced free of charge within the first 30 days.
[3] Refunds are processed within 5-7 business days after the item is received.

Question: How fast will I get my refund?
Answer:


In [6]:
# Without RAG: the model answers from memory (may be wrong / generic).
no_rag_prompt = ("What is the refund timing for Acme? Answer from your own knowledge.")
print("NO-RAG answer :", llm(no_rag_prompt))

# With RAG: answer grounded in the retrieved context.
keys, grounded_prompt = answer("How fast will I get my refund?")
rag_answer = llm(grounded_prompt)
print("RAG answer    :", rag_answer)
print("Grounded in   :", keys)


NO-RAG answer : I’m sorry, but I can’t help with that.


RAG answer    : Refunds are processed within 5‑7 business days after the item is received. [3]
Grounded in   : ['shipping', 'replacement', 'refund']


## 10. What RAG Does NOT Fix
RAG reduces hallucination but does not eliminate it - if retrieval returns irrelevant chunks, the answer is grounded in garbage. That is why the rest of this phase focuses on **making retrieval good** before worrying about the LLM.


## Common Mistakes

- Building RAG when the data fits in the context window and queries are simple.
- Assuming RAG eliminates hallucination entirely.
- Ignoring retrieval quality and blaming the LLM.
- Over-engineering retrieval before validating chunking and embeddings.
- Not evaluating retrieval and generation separately.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Model still hallucinates despite RAG | Retrieved docs are irrelevant | Print top-k chunks; fix retrieval first |
| Answers are generic | Context is too large/noisy | Reduce k, add reranking |
| Model ignores retrieved context | Prompt lacks grounding instructions | Add explicit grounding + citations |
| "This isn't in our docs" | Data not indexed / wrong chunking | Verify ingestion + chunking |

## Best Practices

- Start with the simplest possible RAG and iterate.
- Always evaluate retrieval quality separately from answer quality.
- Log what was retrieved alongside the answer.
- Design for citations from day one.
- Measure latency and cost, not just accuracy.

## Hands-On Practice

1. **Basic:** Ask the mock LLM a question about a private policy; observe the hallucination.
2. **Guided:** Add the policy text to the prompt; observe the improvement.
3. **Independent:** Build a file-based retrieval that finds relevant paragraphs.
4. **Realistic:** Compare RAG vs no-RAG on 10 questions from real documentation.
5. **Challenge:** Design a decision tree for RAG vs fine-tuning vs long-context.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
